# Microsoft Tay: When a Live Feedback Loop Has No Gate

In this notebook I look at Microsoft's Tay chatbot as an eighth mechanism behind an AI ethics failure, and yet another one that doesn't fit the disparate-outcome shape of the first six notebooks or the consent-and-scale shape of the seventh.

Where the Clearview notebook asked whether anyone consented to being enrolled in a system, this one asks a different question again: what happens when a system is designed to learn from public input and publish its own generated output in real time, with nothing sitting between the two. Tay's failure didn't require any single bad decision by Microsoft, a bad threshold, a bad label, a bad proxy. It required an ordinary design choice, learn from your users, made without a second, less ordinary one: bound how much any small group of those users can move what the system learns.

In this notebook, I will:

- Summarize what happened to Tay and how quickly it happened
- Simulate a stream of interactions where a small coordinated group can flood a shared, real-time memory the bot learns from
- Measure how that contamination reaches bystanders who never interacted with the coordinated group at all
- Try an exact-phrase blocklist, the fix that looks obvious first, and see why it barely helps
- Find the fix that actually addresses the mechanism: bounding how much influence any one narrow burst of interaction, from however many accounts, can have on what the bot learns

## 1. Background: What Was Reported

On March 23, 2016, Microsoft launched Tay, a Twitter chatbot built by its Technology and Research and Bing teams, designed to sound like a casual, joking American teenager and, distinctively, to learn from the conversations people had with it and adjust its future replies accordingly. Tay also had a built-in "repeat after me" feature: asked to repeat a phrase, it would tweet that phrase back verbatim.

Within about sixteen hours, groups of users, organized in part on forums like 4chan, discovered both of these features and used them together, feeding Tay a fast, concentrated stream of racist and antisemitic statements, through the repeat function and through ordinary conversation, at a volume and pace that overwhelmed the ordinary mix of people talking to her. Tay began generating and posting hateful content on her own, unprompted, including praise for Hitler and Holocaust denial, to an audience of everyone following the account, not just the people who had provoked her. Microsoft suspended Tay's account within sixteen hours of launch, deleted the great majority of the offensive tweets, and published an apology acknowledging it had not anticipated this kind of coordinated attempt to exploit the bot's learning. A successor bot released later, Zo, was built with much tighter restrictions on what topics it would engage with at all.

## 2. Why This Is a Different Mechanism

Tay never treated any group of people unequally, and nobody scraped anyone's data without consent. Her failure came from a different place: a system that learns continuously from open public input, and publishes what it generates immediately and irreversibly, has no gap in time or scrutiny between an adversarial input arriving and the system's own output reaching an audience of onlookers who had nothing to do with that input. A small, coordinated group exploiting that gap doesn't need to fool a classifier or find a proxy feature, it only needs to out-volume and out-pace the ordinary mix of interactions the system was built to learn from, in a short enough window.

## 3. A Note on the Simulation Below

This notebook does not use Tay's actual training process, and it does not generate or reproduce any of the hateful content real users fed her or that she ended up posting, that content isn't something this notebook needs to recreate to make its point. Instead I represent every simulated message with a single toxicity number between 0 and 1, and represent Tay's real behavior, generating a reply influenced by a rolling window of recent interactions, as a function of the toxicity of those recent messages. What matters here is the mechanism, how a burst of high numbers can dominate that rolling window and change what an unrelated bystander receives next, not the specific words involved.

## 4. Simulating an Ordinary Interaction Stream

I generate a long stream of ordinary interactions. The overwhelming majority of real traffic to a bot like this is everyday, harmless conversation, which I represent as a toxicity value near zero with a small amount of natural noise, each one from a different, unique user.

In [ ]:
import random

random.seed(6)

N_BENIGN = 2000


def make_benign_stream(n):
    """Returns a list of interaction dicts from unique, ordinary users with near-zero toxicity."""
    stream = []
    for i in range(n):
        stream.append({
            "user_id": f"user_{i}",
            "toxicity": max(0.0, random.gauss(0.05, 0.05)),
        })
    return stream


benign_stream = make_benign_stream(N_BENIGN)

## 5. Inserting a Coordinated Brigade

Real accounts describe a small, coordinated group discovering Tay's learning behavior and her repeat-after-me feature, and deliberately flooding her with a fast, concentrated burst of highly toxic messages, far more messages per account in a short stretch of time than an ordinary user would ever send. I insert a burst like that partway through the stream: twenty accounts, each sending fifteen highly toxic messages in quick succession.

In [ ]:
N_BRIGADE_ACCOUNTS = 20
MESSAGES_PER_ACCOUNT = 15


def make_brigade_burst(n_accounts, messages_per_account):
    """Returns a burst of highly toxic interactions from a small number of coordinated accounts."""
    burst = []
    for account in range(n_accounts):
        for _ in range(messages_per_account):
            burst.append({
                "user_id": f"brigade_{account}",
                "toxicity": max(0.0, random.gauss(0.95, 0.05)),
            })
    return burst


brigade_burst = make_brigade_burst(N_BRIGADE_ACCOUNTS, MESSAGES_PER_ACCOUNT)

BRIGADE_START = 1000
timeline = benign_stream[:BRIGADE_START] + brigade_burst + benign_stream[BRIGADE_START:]

## 6. Modeling How Tay Generates a Reply

Tay's own description of herself was that she got "smarter" by learning from her conversations. I model that as a rolling memory: her reply to whoever she's talking to next is influenced by the average toxicity of a fixed number of the most recent messages she has seen, from anyone at all, not just from the specific person she's currently replying to.

In [ ]:
MEMORY_WINDOW = 50


def rolling_reply_toxicity(timeline, window):
    """Returns a list of Tay's outgoing reply toxicity values, one per incoming message."""
    replies = []
    for i in range(len(timeline)):
        recent = timeline[max(0, i - window + 1): i + 1]
        avg_recent_toxicity = sum(m["toxicity"] for m in recent) / len(recent)
        replies.append(avg_recent_toxicity)
    return replies


replies = rolling_reply_toxicity(timeline, MEMORY_WINDOW)

## 7. Measuring the Impact on Bystanders

I check what an ordinary user talking to Tay actually received, comparing the share of toxic replies, above a fixed threshold, in the stretch right before the brigade to the stretch immediately after it, even though those later benign users sent Tay nothing but ordinary messages themselves.

In [ ]:
TOXIC_REPLY_THRESHOLD = 0.3


def toxic_reply_rate(replies, start, end):
    """Returns the fraction of replies in a range that cross the toxic-reply threshold."""
    window = replies[start:end]
    toxic = [r for r in window if r > TOXIC_REPLY_THRESHOLD]
    return len(toxic) / len(window)


before_rate = toxic_reply_rate(replies, 0, BRIGADE_START)
after_start = BRIGADE_START + len(brigade_burst)
after_rate = toxic_reply_rate(replies, after_start, after_start + 200)

print("Toxic reply rate to bystanders, before the brigade:", round(before_rate, 3))
print("Toxic reply rate to bystanders, right after the brigade:", round(after_rate, 3))

## 8. Trying the Fix That Looks Obvious First

The first fix anyone watching this happen might reach for is an exact-phrase blocklist: catch any output that matches a list of known bad phrases and swap in a safe, generic reply instead. I simulate that by blocking the specific replies triggered by messages that match a known, already-flagged set of phrases, while leaving paraphrased or improvised toxic input, the kind a coordinated group generates in real time, untouched.

In [ ]:
def apply_exact_blocklist(timeline, replies, seed):
    """Returns reply toxicity values after blocking only the replies directly triggered by known phrases."""
    rng = random.Random(seed)
    filtered = []
    for message, reply in zip(timeline, replies):
        is_known_phrase = message["toxicity"] > 0.9 and rng.random() < 0.4
        filtered.append(0.05 if is_known_phrase else reply)
    return filtered


filtered_replies = apply_exact_blocklist(timeline, replies, seed=3)

before_rate_filtered = toxic_reply_rate(filtered_replies, 0, BRIGADE_START)
after_rate_filtered = toxic_reply_rate(filtered_replies, after_start, after_start + 200)

print("Toxic reply rate to bystanders, before the brigade, with a blocklist:", round(before_rate_filtered, 3))
print("Toxic reply rate to bystanders, right after the brigade, with a blocklist:", round(after_rate_filtered, 3))